# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

In [ ]:
!apt remove python3-libtorrent -y
!python3 -m pip uninstall libtorrent lbry-libtorrent -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following package was automatically installed and is no longer required:
  libtorrent-rasterbar2.0
Use 'apt autoremove' to remove it.
The following packages will be REMOVED:
  python3-libtorrent
0 upgraded, 0 newly installed, 1 to remove and 35 not upgraded.
After this operation, 3,437 kB disk space will be freed.
(Reading database ... 126450 files and directories currently installed.)
Removing python3-libtorrent (2.0.5-5) ...
Found existing installation: lbry-libtorrent 1.2.4
Uninstalling lbry-libtorrent-1.2.4:
  Successfully uninstalled lbry-libtorrent-1.2.4


In [ ]:
!python3 -m pip install "libtorrent[all]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 63.0 MB/s  0:00:00


### Install libtorrent and Initialize Session

In [ ]:
!apt install python3-libtorrent
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install lbry-libtorrent

import libtorrent as lt

ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  python3-libtorrent
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 597 kB of archives.
After this operation, 3,437 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 python3-libtorrent amd64 2.0.5-5 [597 kB]
Fetched 597 kB in 2s (348 kB/s)
Selecting previously unselected package python3-libtorrent.
(Reading database ... 126442 files and directories currently installed.)
Preparing to unpack .../python3-libtorrent_2.0.5-5_amd64.deb ...
Unpacking python3-libtorrent (2.0.5-5) ...
Setting up python3-libtorrent (2.0.5-5) ...
  Using cached lbry_libtorrent-1.2.4-py3-none-any.whl.metadata (347 bytes)
Using cached lbry_libtorrent-1.2.4-py3-none-any.whl (2.4 MB)


/tmp/ipython-input-3213100306.py:8: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


### Mount Google Drive
To stream files we need to mount Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [ ]:
from google.colab import files

source = files.upload()
params = {
    "save_path": "/content/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

Saving BCA6DA7B8DEC7E3B4A57E6340080603D0F975FB9.torrent to BCA6DA7B8DEC7E3B4A57E6340080603D0F975FB9.torrent


### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)


FloatSlider(value=0.0, disabled=True, layout=Layout(width='auto'), step=0.01, style=SliderStyle(description_wi…

/tmp/ipython-input-3737971044.py:30: DeprecationWarning: is_seed() is deprecated
  if not download.is_seed():
/tmp/ipython-input-3737971044.py:35: DeprecationWarning: name() is deprecated
  download.name(),


KeyboardInterrupt: 

Move file

In [ ]:
!rsync -ah --progress downloads/ onedrive/            #this may take a while